# Query Activation Distribution 分析（单模型）

本 notebook 用于分析单个模型的 query activation 分布，计算 Gini 系数和 Pareto 系数

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from collections import Counter

# 设置字体
rcParams['font.family'] = 'serif'
rcParams['font.serif'] = ['Times New Roman']
rcParams['mathtext.fontset'] = 'stix'
rcParams['font.size'] = 10

%matplotlib inline

: 

## 1. 加载模型和数据集

In [ ]:
# 配置参数
import torch

# ===== 模型和数据配置 =====
MODEL_CONFIG = './configs/dino++/dino++_resnet50_800_1333.py'  # 模型配置文件路径
CHECKPOINT = './checkpoints/your_model.pth'  # 模型权重路径
COCO_PATH = '/path/to/coco'  # COCO 数据集路径
CATEGORY_NAME = 'person'  # 要分析的类别名称，如: 'person', 'car', 'bicycle'
NUM_IMAGES = 100  # 要处理的图片数量（None 表示处理全部）

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 加载模型
from util.lazy_load import Config
from util.utils import load_checkpoint, load_state_dict
from typing import Dict

print("Loading model...")
model = Config(MODEL_CONFIG).model.eval()
checkpoint = load_checkpoint(CHECKPOINT)
if isinstance(checkpoint, Dict) and "model" in checkpoint:
    checkpoint = checkpoint["model"]
load_state_dict(model, checkpoint)
model = model.to(device)
print(f"Model loaded successfully: {model.__class__.__name__}")
print(f"Number of queries: {model.num_queries if hasattr(model, 'num_queries') else 'Unknown'}")

In [ ]:
# 加载 COCO 数据集
from datasets.coco import CocoDetection
from util.collate_fn import collate_fn
from torch.utils.data import DataLoader

print("Loading COCO val2017 dataset...")
dataset = CocoDetection(
    img_folder=f"{COCO_PATH}/val2017",
    ann_file=f"{COCO_PATH}/annotations/instances_val2017.json",
    transforms=None,  # eval_transform is integrated in the model
    train=False,
)

# 获取目标类别的 ID
coco = dataset.coco
category_id = coco.getCatIds(catNms=[CATEGORY_NAME])[0]
print(f"Target category: {CATEGORY_NAME} (ID: {category_id})")

# 获取包含该类别的图片 IDs
img_ids_with_category = coco.getImgIds(catIds=[category_id])
if NUM_IMAGES:
    img_ids_with_category = img_ids_with_category[:NUM_IMAGES]
print(f"Found {len(img_ids_with_category)} images with category '{CATEGORY_NAME}'")

In [ ]:
# 推理并记录 query activation
from tqdm import tqdm

print("\\nRunning inference and collecting query activations...")

# 用于收集所有被激活的 query 索引
all_cat_query_indexes = []

model.eval()
with torch.no_grad():
    for img_id in tqdm(img_ids_with_category, desc="Processing images"):
        # 加载图片和标注
        img_info = coco.loadImgs(img_id)[0]
        img_path = f"{COCO_PATH}/val2017/{img_info['file_name']}"
        
        # 获取图片数据
        image, target = dataset[dataset.ids.index(img_id)]
        images = [image.to(device)]
        
        # 模型推理
        outputs = model(images)
        prediction = outputs[0]
        
        # 获取预测结果
        pred_boxes = prediction['boxes'].cpu()
        pred_labels = prediction['labels'].cpu()
        pred_scores = prediction['scores'].cpu()
        
        # 过滤：只保留目标类别且置信度 > 0.3 的检测
        score_threshold = 0.3
        cat_keep = (pred_labels == category_id) & (pred_scores > score_threshold)
        
        # 记录被激活的 query 索引
        # 注意：这里假设 outputs 中包含 query_indices 或者通过某种方式获取
        # 根据您的模型结构，可能需要调整这部分代码
        if 'query_indices' in prediction:
            selected_query_indices = prediction['query_indices'][cat_keep]
        else:
            # 如果没有 query_indices，使用预测结果的索引
            selected_query_indices = torch.where(cat_keep)[0]
        
        all_cat_query_indexes.extend(selected_query_indices.tolist())

print(f\"\\nTotal activations collected: {len(all_cat_query_indexes)}\")\nprint(f\"Unique queries activated: {len(set(all_cat_query_indexes))}\")

In [ ]:
# 统计每个 query 的激活次数
activation_counts = Counter(all_cat_query_indexes)

print(f"\nQuery activation statistics:")
print(f"  Unique queries activated: {len(activation_counts)}")
print(f"  Total activations: {sum(activation_counts.values())}")
print(f"\nTop 20 most activated queries:")
for query_idx, count in activation_counts.most_common(20):
    print(f"  Query {query_idx:3d}: {count:4d} times")

print(f"\nQuery indices distribution: {activation_counts}")

## 2. 定义 Gini 和 Pareto 系数计算函数

In [ ]:
def calculate_gini_coefficient(activation_counts):
    """
    计算 Gini 系数
    
    Args:
        activation_counts: dict or Counter, {query_index: activation_count}
        
    Returns:
        float: Gini coefficient (0-1)，值越大表示分布越不均匀
    """
    if len(activation_counts) == 0:
        return 0.0
    
    # 获取所有激活次数
    values = np.array(list(activation_counts.values()), dtype=float)
    values = values[values > 0]
    
    if len(values) == 0:
        return 0.0
    
    # 排序
    sorted_values = np.sort(values)
    n = len(sorted_values)
    
    # 计算 Gini 系数: G = (2 * sum(i * x_i)) / (n * sum(x_i)) - (n+1)/n
    cumsum = np.cumsum(sorted_values)
    gini = (2 * np.sum((np.arange(1, n + 1) * sorted_values))) / (n * cumsum[-1]) - (n + 1) / n
    
    return gini


def calculate_pareto_coefficient(activation_counts, threshold_percentile=20):
    """
    计算 Pareto 系数 (80-20 rule)
    
    Args:
        activation_counts: dict or Counter
        threshold_percentile: int, 百分位数，默认 20 表示计算前 20% 的 queries
        
    Returns:
        float: Pareto coefficient，表示前 threshold_percentile% 的 queries 产生的激活占比
    """
    if len(activation_counts) == 0:
        return 0.0
    
    values = np.array(list(activation_counts.values()), dtype=float)
    values = values[values > 0]
    
    if len(values) == 0:
        return 0.0
    
    # 排序（降序）
    sorted_values = np.sort(values)[::-1]
    
    # 计算累积和
    cumsum = np.cumsum(sorted_values)
    total = cumsum[-1]
    
    # 前 threshold_percentile% 的 queries 产生的 activation 占比
    n_top = max(1, int(len(sorted_values) * threshold_percentile / 100))
    top_contribution = cumsum[n_top - 1] / total
    
    return top_contribution

## 3. 定义绘图函数

In [ ]:
def plot_query_activation_distribution(activation_counts, model_name, category_name, save_dir='results/activation_analysis'):
    """
    绘制单个模型的 Query Activation Distribution 图
    
    Args:
        activation_counts: dict or Counter, {query_index: activation_count}
        model_name: str, 模型名称
        category_name: str, 类别名称
        save_dir: str, 保存目录
    """
    os.makedirs(save_dir, exist_ok=True)
    
    if len(activation_counts) == 0:
        print(f"Warning: No activation data for {model_name} - {category_name}")
        return
    
    # 计算 Gini 和 Pareto 系数
    gini = calculate_gini_coefficient(activation_counts)
    pareto = calculate_pareto_coefficient(activation_counts, threshold_percentile=20)
    
    # 获取激活次数并排序（降序）
    counts = np.array(list(activation_counts.values()))
    counts_sorted = np.sort(counts)[::-1]
    
    # 创建图表
    plt.figure(figsize=(5, 3.5))
    
    # 绘制激活分布（对数坐标）
    x = np.arange(len(counts_sorted))
    plt.plot(x, counts_sorted, 'b-', linewidth=1.5, alpha=0.8)
    plt.scatter(x[::max(1, len(x)//20)], counts_sorted[::max(1, len(x)//20)], 
                c='blue', s=20, alpha=0.6, zorder=5)
    
    plt.yscale('log')
    plt.xlabel('Query Index (Ranked)', fontsize=11)
    plt.ylabel('Activation Count (log)', fontsize=11)
    
    # 标题包含 Gini 和 Pareto 系数
    title = f'{model_name} - {category_name}\n'
    title += f'Gini: {gini:.3f}, Pareto(20%): {pareto:.3f}'
    plt.title(title, fontsize=11, pad=10)
    
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()
    
    # 保存图片
    save_path = os.path.join(save_dir, f'activation_dist_{model_name}_{category_name}.pdf')
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    print(f"Saved: {save_path}")
    
    plt.show()
    plt.close()
    
    # 打印统计信息
    print(f"\nStatistics for {model_name} - {category_name}:")
    print(f"  Total unique queries: {len(activation_counts)}")
    print(f"  Total activations: {sum(activation_counts.values())}")
    print(f"  Gini coefficient: {gini:.4f}")
    print(f"  Pareto coefficient (20%): {pareto:.4f}")
    print(f"  Max activation count: {max(activation_counts.values())}")
    print(f"  Mean activation count: {np.mean(counts):.2f}")
    print(f"  Median activation count: {np.median(counts):.2f}")

In [ ]:
# 多模型对比功能已移除
# 本 notebook 现在只专注于单模型分析
# 如需对比多个模型，请多次调用 plot_query_activation_distribution()

## 4. 绘制 Query Activation Distribution 图

In [ ]:
# 使用之前收集的数据绘制图表
# activation_counts 已经在上面的步骤中计算好了

# 如果您想测试绘图功能但还没有运行推理，可以使用模拟数据：
# np.random.seed(42)
# probs = np.array([1/(i+1)**2.0 for i in range(900)])
# probs = probs / probs.sum()
# sampled_queries = np.random.choice(900, size=5000, p=probs)
# activation_counts = Counter(sampled_queries.tolist())

In [ ]:
# 绘制 Query Activation Distribution 图
model_name = MODEL_CONFIG.split('/')[-2]  # 从配置文件路径提取模型名称
plot_query_activation_distribution(
    activation_counts=activation_counts,
    model_name=model_name,
    category_name=CATEGORY_NAME
)

## 5. (可选) 使用模拟数据测试

如果您想在不运行完整推理的情况下测试绘图功能

In [ ]:
# 创建模拟数据用于测试
np.random.seed(42)

# 模拟一个不均匀分布（类似真实的 DETR 模型）
probs = np.array([1/(i+1)**2.0 for i in range(900)])
probs = probs / probs.sum()

sampled_queries = np.random.choice(900, size=5000, p=probs)
test_activation_counts = Counter(sampled_queries.tolist())

print(f"模拟数据已创建")
print(f"激活的唯一 queries: {len(test_activation_counts)}")
print(f"总激活次数: {sum(test_activation_counts.values())}")

# 绘制测试图
plot_query_activation_distribution(
    activation_counts=test_activation_counts,
    model_name='Test-Model',
    category_name='test'
)

## 6. (可选) 批量处理多个类别

In [ ]:
# 如果需要为多个类别分别绘制图表：
#
# category_data = {
#     'person': Counter(person_query_indexes),
#     'car': Counter(car_query_indexes),
#     'bicycle': Counter(bicycle_query_indexes),
# }
#
# model_name = 'DINO'
# for category_name, activation_counts in category_data.items():
#     plot_query_activation_distribution(
#         activation_counts=activation_counts,
#         model_name=model_name,
#         category_name=category_name
#     )

print("批量处理示例（如需使用请取消注释）")